
# Pose Quality Check + Preprocessing Pipeline 

This notebook performs two stages:

1. **QC (Quality Control)** — counts and lists 'bad' windows for each metric.
2. **Preprocessing** — aggregates OpenPose keypoints into regions, then **masks only** those metric-windows flagged as bad by QC.

**How to use:** Set everything in the first **Parameters** cell, then run cells in order.


In [ ]:

# ============================ PARAMETERS (EDIT ME) ============================
# High-level switch (optional): choose data blocks to process
MODE = "baseline"   # options: "baseline", "experimental"

# I/O directories (override if you want explicit paths)
# Pose CSVs for QC and RAW input for preprocessing
DIRS = {
    "baseline": {
        "qc_output": "data/qc_outputs_bsl",
        "raw_input": "data/raw_pose/baseline_pose",    # used in preprocessing stage, and QC
        "preproc_output": "data/preprocessed_pose/baseline_pose",
    },
    "experimental": {
        "qc_output": "data/qc_outputs_exp",
        "raw_input": "data/raw_pose/experimental_pose",
        "preproc_output": "data/preprocessed_pose/experimental_pose",
    },
}

# --- QC config ---
WINDOW_SIZE = 1800            # frames per window (e.g., 60s * 30fps)
OVERLAP = 0.0                 # fractional overlap (0.0 = none)
CONFIDENCE_THRESHOLD = 0.3    # min prob for a keypoint to be "present"
MAX_INTERP = 60               # max allowed consecutive missing frames in a window

# --- Preprocessing config ---
# QC window spec MUST match QC stage or spans will be wrong.
QC_WINDOW_FRAMES = WINDOW_SIZE
QC_OVERLAP = OVERLAP

# Map QC metric labels → columns to blank (set NaN) during preprocessing
# Keys must align with QC metric names (case-insensitive).
QC_TO_COLUMNS = {
    "eyes": [
        "blink_dist",
        "left_eye_x", "left_eye_y", "left_eye_prob", "left_eye_magnitude",
        "right_eye_x", "right_eye_y", "right_eye_prob", "right_eye_magnitude",
    ],
    "head_rotation": [
        "head_rotation_angle",
    ],
    "mouth_dist": [
        "mouth_dist",
    ],
    "pupils_combined": [
        "avg_pupil_x", "avg_pupil_y", "avg_pupil_magnitude",
        "left_pupil_x", "left_pupil_y", "left_pupil_prob", "left_pupil_magnitude",
        "right_pupil_x", "right_pupil_y", "right_pupil_prob", "right_pupil_magnitude",
    ],
    "center_face": [
        "center_face_x", "center_face_y", "center_face_prob", "center_face_magnitude",
    ],
}
# ============================================================================

# ---- Derived paths from MODE ----
INPUT_DIR_QC = DIRS[MODE]["raw_input"]  
OUTPUT_DIR_QC = DIRS[MODE]["qc_output"]
RAW_INPUT_DIR = DIRS[MODE]["raw_input"]
PREPROC_OUTPUT_DIR = DIRS[MODE]["preproc_output"]
QC_BAD_IDX_FILE = f"{OUTPUT_DIR_QC}/metric_bad_window_indices.csv"

# Sanity echo
print("MODE:", MODE)
print("QC input:", INPUT_DIR_QC)
print("QC output:", OUTPUT_DIR_QC)
print("RAW input (preproc):", RAW_INPUT_DIR)
print("Preproc output:", PREPROC_OUTPUT_DIR)
print("QC bad-index file (will be created by QC, read by Preproc):", QC_BAD_IDX_FILE)


MODE: baseline
QC input: data/raw_pose/baseline_pose
QC output: data/qc_outputs_bsl
RAW input (preproc): data/raw_pose/baseline_pose
Preproc output: data/preprocessed_pose/baseline_pose
QC bad-index file (will be created by QC, read by Preproc): data/qc_outputs_bsl/metric_bad_window_indices.csv


In [4]:

import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm

# Mapping of metrics to the keypoint indices they depend on (for QC stage)
METRIC_KPS = {
    "eyes":            [37, 38, 40, 41, 43, 44, 46, 47],  # blink-related kps
    "head_rotation":   [36, 45],
    "mouth_dist":      [62, 66],
    "pupils_combined": [68, 69],
    "center_face":     list(range(27, 36)),  # nose/center face landmarks
}
RELEVANT_KPS = sorted({kp for kps in METRIC_KPS.values() for kp in kps})


# ---------------- QC: Utility Functions ----------------

def window_ranges(n_rows: int, window_size: int, overlap: float):
    """
    Given the number of rows, return (start, end) windows.
    If file shorter than a window, return [].
    """
    if n_rows < window_size:
        return []
    step = max(1, int(round(window_size * (1 - overlap))))
    return [(s, s + window_size) for s in range(0, n_rows - window_size + 1, step)]

def _max_true_run_length(b: pd.Series) -> int:
    """Longest consecutive run of True in a boolean series."""
    run = max_run = 0
    for v in b.to_numpy():
        if v:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

def kp_missing_series(df_w: pd.DataFrame, i: int, conf_thresh: float) -> pd.Series:
    """
    True means keypoint i is missing in that frame:
      - prob < conf_thresh OR x/y is NaN
    If columns missing, treat as fully missing.
    """
    xcol, ycol, pcol = f"x{i}", f"y{i}", f"prob{i}"
    if (xcol not in df_w.columns) or (ycol not in df_w.columns) or (pcol not in df_w.columns):
        return pd.Series(True, index=df_w.index)
    good = (df_w[pcol] >= conf_thresh) & df_w[xcol].notna() & df_w[ycol].notna()
    return ~good


# ---------------- QC: Core Analysis ----------------

def analyze_file_qc(fp: str, window_size: int, overlap: float, conf_thresh: float, max_interp: int):
    """
    For one CSV:
      - Slice into windows
      - 'Bad' keypoints: longest missing run > max_interp
      - 'Bad' metric: ANY of its keypoints bad
      - Return 3 DataFrames: keypoint summary, metric summary, metric bad-window indices
    """
    df = pd.read_csv(fp)
    base = os.path.basename(fp)
    wranges = window_ranges(len(df), window_size, overlap)
    total_windows = len(wranges)

    if total_windows == 0:
        kp_rows = [{"file": base, "keypoint": i, "bad_windows": 0, "total_windows": 0, "pct_bad": np.nan} for i in RELEVANT_KPS]
        met_rows = [{"file": base, "metric": m, "bad_windows": 0, "total_windows": 0, "pct_bad": np.nan} for m in METRIC_KPS.keys()]
        return pd.DataFrame(kp_rows), pd.DataFrame(met_rows), pd.DataFrame([])

    kp_bad_counts = {i: 0 for i in RELEVANT_KPS}
    metric_bad_counts = {m: 0 for m in METRIC_KPS}
    metric_bad_indices = {m: [] for m in METRIC_KPS}

    for win_idx, (s, e) in enumerate(wranges):
        df_w = df.iloc[s:e].reset_index(drop=True)

        # Keypoint-level QC
        kp_bad = {}
        for i in RELEVANT_KPS:
            missing = kp_missing_series(df_w, i, conf_thresh)
            longest_gap = _max_true_run_length(missing)
            is_bad = (longest_gap > max_interp)
            kp_bad[i] = is_bad
            if is_bad:
                kp_bad_counts[i] += 1

        # Metric-level QC
        for m, kps in METRIC_KPS.items():
            if any(kp_bad.get(i, True) for i in kps):
                metric_bad_counts[m] += 1
                metric_bad_indices[m].append((win_idx, s, e))

    # Build outputs
    kp_rows = []
    for i in RELEVANT_KPS:
        bw = kp_bad_counts[i]
        kp_rows.append({"file": base, "keypoint": i, "bad_windows": bw, "total_windows": total_windows, "pct_bad": bw / total_windows})

    met_rows = []
    for m in METRIC_KPS:
        bw = metric_bad_counts[m]
        met_rows.append({"file": base, "metric": m, "bad_windows": bw, "total_windows": total_windows, "pct_bad": bw / total_windows})

    met_idx_rows = []
    for m, entries in metric_bad_indices.items():
        for (win_idx, s, e) in entries:
            met_idx_rows.append({"file": base, "metric": m, "window_index": win_idx, "start_frame": s, "end_frame_exclusive": e})

    return pd.DataFrame(kp_rows), pd.DataFrame(met_rows), pd.DataFrame(met_idx_rows)


def run_qc(input_dir: str, output_dir: str, window_size: int, overlap: float, conf_thresh: float, max_interp: int):
    """Scan all CSVs in input_dir; write QC outputs to output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv")]
    all_kp, all_met, all_met_idx = [], [], []

    for f in tqdm(files, desc="QC scanning"):
        fp = os.path.join(input_dir, f)
        try:
            kp_df, met_df, met_idx_df = analyze_file_qc(fp, window_size, overlap, conf_thresh, max_interp)
        except Exception as e:
            kp_df = pd.DataFrame([{"file": f, "keypoint": None, "bad_windows": None, "total_windows": None, "pct_bad": None, "error": str(e)}])
            met_df = pd.DataFrame([{"file": f, "metric": None, "bad_windows": None, "total_windows": None, "pct_bad": None, "error": str(e)}])
            met_idx_df = pd.DataFrame([{"file": f, "metric": None, "window_index": None, "start_frame": None, "end_frame_exclusive": None, "error": str(e)}])

        all_kp.append(kp_df)
        all_met.append(met_df)
        all_met_idx.append(met_idx_df)

    out_kp = pd.concat(all_kp, ignore_index=True)
    out_met = pd.concat(all_met, ignore_index=True)
    out_met_idx = pd.concat(all_met_idx, ignore_index=True)

    out_kp_path = os.path.join(output_dir, "keypoint_bad_windows.csv")
    out_met_path = os.path.join(output_dir, "metric_bad_windows.csv")
    out_met_idx_path = os.path.join(output_dir, "metric_bad_window_indices.csv")

    out_kp.to_csv(out_kp_path, index=False)
    out_met.to_csv(out_met_path, index=False)
    out_met_idx.to_csv(out_met_idx_path, index=False)

    print("Wrote:")
    print(" ", out_kp_path)
    print(" ", out_met_path)
    print(" ", out_met_idx_path)

    return out_kp_path, out_met_path, out_met_idx_path


def summarize_bad_windows(path: str):
    """Quick aggregation helper for QC outputs."""
    df = pd.read_csv(path)
    total_bad = df['bad_windows'].sum()
    total_windows = df['total_windows'].sum()
    pct = total_bad / total_windows * 100 if total_windows > 0 else float('nan')
    return total_bad, total_windows, pct




In [5]:

# ---------------- Run QC ----------------
out_kp_p, out_met_p, out_met_idx_p = run_qc(
    input_dir=INPUT_DIR_QC,
    output_dir=OUTPUT_DIR_QC,
    window_size=WINDOW_SIZE,
    overlap=OVERLAP,
    conf_thresh=CONFIDENCE_THRESHOLD,
    max_interp=MAX_INTERP,
)

# Optional: print a quick summary
total_bad, total_win, pct_bad = summarize_bad_windows(out_kp_p)
print(f"QC summary: {total_bad}/{total_win} windows bad ({pct_bad:.2f}%)")


QC scanning: 100%|██████████| 215/215 [00:14<00:00, 15.15it/s]

Wrote:
  data/qc_outputs_bsl/keypoint_bad_windows.csv
  data/qc_outputs_bsl/metric_bad_windows.csv
  data/qc_outputs_bsl/metric_bad_window_indices.csv
QC summary: 55/19826 windows bad (0.28%)


In [6]:

# ---------------- Preprocessing: Helpers ----------------

def compute_averages(df: pd.DataFrame, conf_thresh: float) -> pd.DataFrame:
    """
    Collapse raw keypoints into mean positions + mean confidence for regions.
    If confidence is low, mark positions as NaN, then interpolate.
    """
    averaged_df = pd.DataFrame({
        # Center-face region (27–35)
        'center_face_x': df[[f'x{i}' for i in range(27, 36)]].mean(axis=1),
        'center_face_y': df[[f'y{i}' for i in range(27, 36)]].mean(axis=1),
        'center_face_prob': df[[f'prob{i}' for i in range(27, 36)]].mean(axis=1),

        # Eyes (36–41 left, 42–47 right)
        'left_eye_x': df[[f'x{i}' for i in range(36, 42)]].mean(axis=1),
        'left_eye_y': df[[f'y{i}' for i in range(36, 42)]].mean(axis=1),
        'left_eye_prob': df[[f'prob{i}' for i in range(36, 42)]].mean(axis=1),

        'right_eye_x': df[[f'x{i}' for i in range(42, 48)]].mean(axis=1),
        'right_eye_y': df[[f'y{i}' for i in range(42, 48)]].mean(axis=1),
        'right_eye_prob': df[[f'prob{i}' for i in range(42, 48)]].mean(axis=1),

        # Pupils (68, 69)
        'left_pupil_x': df.get('x68', pd.Series(np.nan, index=df.index)),
        'left_pupil_y': df.get('y68', pd.Series(np.nan, index=df.index)),
        'left_pupil_prob': df.get('prob68', pd.Series(np.nan, index=df.index)),

        'right_pupil_x': df.get('x69', pd.Series(np.nan, index=df.index)),
        'right_pupil_y': df.get('y69', pd.Series(np.nan, index=df.index)),
        'right_pupil_prob': df.get('prob69', pd.Series(np.nan, index=df.index)),
    })

    # If confidence below threshold, treat positions as missing
    for part in ['center_face', 'left_eye', 'right_eye', 'left_pupil', 'right_pupil']:
        prob_col = f'{part}_prob'
        for axis in ['x', 'y']:
            val_col = f'{part}_{axis}'
            averaged_df.loc[averaged_df[prob_col] < conf_thresh, val_col] = np.nan

    # Fill short gaps by interpolation
    averaged_df.interpolate(method='linear', limit_direction='both', inplace=True)
    return averaged_df

def compute_magnitude(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each *_x with matching *_y, add *_magnitude = frame-to-frame displacement.
    displacement = sqrt((Δx)^2 + (Δy)^2).
    """
    for col in list(df.columns):
        if col.endswith('_x'):
            y_col = col.replace('_x', '_y')
            if y_col in df.columns:
                mag_col = col.replace('_x', '_magnitude')
                dx = df[col].diff()
                dy = df[y_col].diff()
                df[mag_col] = np.sqrt(dx**2 + dy**2)
                # first row has no previous frame → set to 0
                df.loc[df.index[0], mag_col] = 0.0
    return df

def compute_combined_eyes(df: pd.DataFrame) -> pd.DataFrame:
    """Average pupil positions and magnitude."""
    df['avg_pupil_x'] = (df['left_pupil_x'] + df['right_pupil_x']) / 2
    df['avg_pupil_y'] = (df['left_pupil_y'] + df['right_pupil_y']) / 2
    df['avg_pupil_magnitude'] = np.sqrt(df['avg_pupil_x']**2 + df['avg_pupil_y']**2)
    return df

def compute_blink(df_raw: pd.DataFrame) -> pd.Series:
    """Blink proxy: average vertical eyelid distances (both eyes)."""
    top_right_x = df_raw[['x37', 'x38']].mean(axis=1)
    top_right_y = df_raw[['y37', 'y38']].mean(axis=1)
    bottom_right_x = df_raw[['x40', 'x41']].mean(axis=1)
    bottom_right_y = df_raw[['y40', 'y41']].mean(axis=1)
    right_eye_dist = np.sqrt((top_right_x - bottom_right_x)**2 + (top_right_y - bottom_right_y)**2)

    top_left_x = df_raw[['x43', 'x44']].mean(axis=1)
    top_left_y = df_raw[['y43', 'y44']].mean(axis=1)
    bottom_left_x = df_raw[['x46', 'x47']].mean(axis=1)
    bottom_left_y = df_raw[['y46', 'y47']].mean(axis=1)
    left_eye_dist = np.sqrt((top_left_x - bottom_left_x)**2 + (top_left_y - bottom_left_y)**2)

    return (right_eye_dist + left_eye_dist) / 2.0

def compute_head_rotation(df_raw: pd.DataFrame) -> pd.Series:
    """Head rotation angle (radians) from keypoints 36→45."""
    dx = df_raw['x45'] - df_raw['x36']
    dy = df_raw['y45'] - df_raw['y36']
    return np.arctan2(dy, dx)

def compute_mouth_distance(df_raw: pd.DataFrame) -> pd.Series:
    """Mouth opening proxy (distance between keypoints 62 and 66)."""
    return np.sqrt((df_raw['x62'] - df_raw['x66'])**2 + (df_raw['y62'] - df_raw['y66'])**2)


# ---------------- Preprocessing: QC Masking Core ----------------

def load_bad_windows(bad_idx_csv: str, qc_window_frames: int, qc_overlap: float):
    """
    Build a dictionary:
        bad_map[file_basename][metric_lower] = list of (start_frame, end_frame_exclusive)
    Accept explicit spans or window_index (rebuild spans with QC window spec).
    """
    if not os.path.isfile(bad_idx_csv):
        sys.exit(f"[FATAL] QC bad-window index CSV not found: {bad_idx_csv}")

    df = pd.read_csv(bad_idx_csv)
    if 'file' not in df.columns or 'metric' not in df.columns:
        sys.exit("[FATAL] QC CSV must include 'file' and 'metric' columns.")

    # Normalize names
    df['file'] = df['file'].astype(str).map(lambda s: os.path.basename(s).strip())
    df['metric'] = df['metric'].astype(str).str.strip().str.lower()

    has_spans = {'start_frame', 'end_frame_exclusive'}.issubset(df.columns)
    has_index = 'window_index' in df.columns

    step_frames = int(round(qc_window_frames * (1.0 - qc_overlap)))

    # Rebuild spans if needed
    if not has_spans and has_index:
        df = df.copy()
        widx = df['window_index'].astype(int)
        df['start_frame'] = (widx * step_frames).astype(int)
        df['end_frame_exclusive'] = (df['start_frame'] + qc_window_frames).astype(int)

    # Note: final window may be truncated
    bad_map = {}
    for (fname, metric), grp in df.groupby(['file', 'metric']):
        spans = list(zip(grp['start_frame'].astype(int), grp['end_frame_exclusive'].astype(int)))
        bad_map.setdefault(fname, {})[metric] = spans

    total = sum(len(v) for d in bad_map.values() for v in d.values())
    print(f"[QC] Loaded {total} bad window span(s) from {bad_idx_csv} (window={qc_window_frames}, step={step_frames}).")
    return bad_map, step_frames


def apply_bad_masks(file_basename: str, df_proc: pd.DataFrame, bad_map: dict, qc_to_columns: dict):
    """
    Write NaNs only where QC flagged windows for specific metrics.
    Returns df_proc (modified), stats (per-metric), total_masked_frames.
    """
    n_frames = len(df_proc)
    stats = {}
    total_masked_frames = 0

    present = bad_map.get(file_basename, {})
    metric_masks = {m: np.zeros(n_frames, dtype=bool) for m in qc_to_columns.keys()}

    # Build per-metric masks
    for qc_metric_raw, windows in present.items():
        qc_metric = qc_metric_raw.lower()
        if qc_metric not in metric_masks:
            continue
        for (s, e) in windows:
            s = max(0, int(s))
            e = min(n_frames, int(e))
            if s < e:
                metric_masks[qc_metric][s:e] = True

    # Apply masks and collect stats
    for qc_metric, mask in metric_masks.items():
        frames_masked = int(mask.sum())
        if frames_masked > 0:
            for col in qc_to_columns[qc_metric]:
                if col in df_proc.columns:
                    df_proc.loc[mask, col] = np.nan
        stats[qc_metric] = {
            "frames_total": n_frames,
            "frames_masked": frames_masked,
            "pct_masked": (frames_masked / n_frames) if n_frames > 0 else np.nan,
            "windows_masked": int(len(present.get(qc_metric, [])))
        }
        total_masked_frames += frames_masked

    return df_proc, stats, total_masked_frames


In [7]:

# ---------------- Run Preprocessing (with QC masking) ----------------

def process_data(input_dir: str, output_dir: str, qc_bad_idx_file: str, qc_to_columns: dict,
                 qc_window_frames: int, qc_overlap: float, conf_thresh: float):
    """
    1) Load QC map and verify filenames overlap.
    2) For each CSV: compute features → apply masks → save → append to drop report.
    3) Save the drop report.
    """
    bad_map, step_frames = load_bad_windows(qc_bad_idx_file, qc_window_frames, qc_overlap)

    proc_files = {os.path.basename(f) for f in os.listdir(input_dir) if f.endswith('.csv')}
    qc_files = set(bad_map.keys())
    intersection = proc_files & qc_files
    if not intersection:
        sys.exit(
            "[FATAL] No filename overlap between QC and RAW_INPUT_DIR.\n"
            "QC files ({len(qc_files)}): {sorted(list(qc_files))[:5]}...\n"
            "RAW files ({len(proc_files))}: {sorted(list(proc_files))[:5]}..."
        )
    print(f"[OK] QC CSV found and {len(intersection)} matching file(s).")

    os.makedirs(output_dir, exist_ok=True)
    csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

    report_rows = []

    for csv_file in tqdm(csv_files, desc="Preprocessing CSV files w/ QC masking"):
        file_path = os.path.join(input_dir, csv_file)
        df_raw = pd.read_csv(file_path)

        # Build preprocessed feature table
        averaged_df = compute_averages(df_raw, conf_thresh)
        averaged_df = compute_magnitude(averaged_df)
        averaged_df = compute_combined_eyes(averaged_df)
        averaged_df['blink_dist'] = compute_blink(df_raw)
        averaged_df['head_rotation_angle'] = compute_head_rotation(df_raw)
        averaged_df['mouth_dist'] = compute_mouth_distance(df_raw)

        # Apply QC masks
        averaged_df, stats, total_masked_frames = apply_bad_masks(
            file_basename=os.path.basename(csv_file),
            df_proc=averaged_df,
            bad_map=bad_map,
            qc_to_columns=qc_to_columns
        )

        # If QC says bad windows exist but nothing masked, fail fast
        if (csv_file in bad_map) and sum(len(v) for v in bad_map[csv_file].values()) > 0 and total_masked_frames == 0:
            sample = []
            for m, spans in bad_map[csv_file].items():
                if spans:
                    s0, e0 = spans[0]
                    sample.append(f"{m}[{s0}:{e0})")
                if len(sample) >= 3:
                    break
            sys.exit(f"[FATAL] 0 NaNs masked for {csv_file} despite QC spans: {', '.join(sample)}. "
                     "Check filename/metric normalization and column names.")

        # Save cleaned CSV
        out_path = os.path.join(output_dir, os.path.splitext(csv_file)[0] + ".csv")
        averaged_df.to_csv(out_path, index=False)

        # Append stats
        for qc_metric, st in stats.items():
            report_rows.append({
                "file": csv_file,
                "qc_metric": qc_metric,
                "frames_total": st["frames_total"],
                "frames_masked": st["frames_masked"],
                "pct_masked": st["pct_masked"],
                "windows_masked": st["windows_masked"]
            })

    # Write drop report
    report_path = os.path.join(output_dir, "preprocess_drop_report.csv")
    pd.DataFrame(report_rows).to_csv(report_path, index=False)
    print(f"Wrote preprocessed files to: {output_dir}")
    print(f"Wrote drop report: {report_path}")

    return report_path


# Execute preprocessing using the parameters above
report_path = process_data(
    input_dir=RAW_INPUT_DIR,
    output_dir=PREPROC_OUTPUT_DIR,
    qc_bad_idx_file=QC_BAD_IDX_FILE,
    qc_to_columns=QC_TO_COLUMNS,
    qc_window_frames=QC_WINDOW_FRAMES,
    qc_overlap=QC_OVERLAP,
    conf_thresh=CONFIDENCE_THRESHOLD,
)


[QC] Loaded 16 bad window span(s) from data/qc_outputs_bsl/metric_bad_window_indices.csv (window=1800, step=1800).
[OK] QC CSV found and 6 matching file(s).


Preprocessing CSV files w/ QC masking: 100%|██████████| 215/215 [00:31<00:00,  6.93it/s]

Wrote preprocessed files to: data/preprocessed_pose/baseline_pose
Wrote drop report: data/preprocessed_pose/baseline_pose/preprocess_drop_report.csv
